In [ ]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 32.2 MB/s eta 0:00:00


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import optuna
import random

# =====================================================
# SpecAugment
# =====================================================

class SpecAugment(nn.Module):
    def __init__(self, freq_mask_param=4, time_mask_param=8):
        super().__init__()
        self.freq_mask_param = freq_mask_param
        self.time_mask_param = time_mask_param

    def forward(self, x):

        if not self.training:
            return x

        x = x.clone()

        B, C, F, T = x.shape

        for i in range(B):

            f = random.randint(0, self.freq_mask_param)
            f0 = random.randint(0, max(0, F - f))
            x[i, :, f0:f0+f, :] = 0

            t = random.randint(0, self.time_mask_param)
            t0 = random.randint(0, max(0, T - t))
            x[i, :, :, t0:t0+t] = 0

        return x

# =====================================================
# Fixed Teacher Architecture
# =====================================================

class KWSCNNTeacherI(nn.Module):
    def __init__(self, num_classes=12, dropout_fc=0.35, freq_mask=4, time_mask=8):
        super().__init__()

        self.specaug = SpecAugment(freq_mask, time_mask)

        self.features = nn.Sequential(

            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),

            nn.Conv2d(16, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),

            nn.MaxPool2d(2),
            nn.Dropout(0.10),

            nn.Conv2d(16, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.MaxPool2d(2),
            nn.Dropout(0.15),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.Dropout(0.20),

            nn.Conv2d(128, 192, kernel_size=3, padding=1),
            nn.BatchNorm2d(192),
            nn.ReLU()
        )

        self.pool = nn.AdaptiveAvgPool2d((1,1))

        self.classifier = nn.Sequential(
            nn.Linear(192, 128),
            nn.ReLU(),
            nn.Dropout(dropout_fc),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):

        x = self.specaug(x)

        x = self.features(x)
        x = self.pool(x)
        x = x.flatten(1)

        return self.classifier(x)

# =====================================================
# Load Data
# =====================================================

data = torch.load("/content/drive/MyDrive/gsc_mfcc_40x98.pt")

X_train = data["X_train"].unsqueeze(1).float()
y_train = data["y_train"]

X_val = data["X_val"].unsqueeze(1).float()
y_val = data["y_val"]

# =====================================================
# Normalize
# =====================================================

mean = X_train.mean()
std = X_train.std()

X_train = (X_train - mean) / std
X_val = (X_val - mean) / std

# =====================================================
# Dataset
# =====================================================

train_ds = torch.utils.data.TensorDataset(X_train, y_train)
val_ds = torch.utils.data.TensorDataset(X_val, y_val)

# =====================================================
# Device
# =====================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =====================================================
# Optuna Objective
# =====================================================

def objective(trial):

    # ---------------- Search Space ----------------
    lr = trial.suggest_float("lr", 1e-4, 3e-3, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-3, log=True)
    label_smoothing = trial.suggest_float("label_smoothing", 0.0, 0.15)
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])

    dropout_fc = trial.suggest_float("dropout_fc", 0.2, 0.5)

    freq_mask = trial.suggest_int("freq_mask", 2, 8)
    time_mask = trial.suggest_int("time_mask", 4, 12)

    # ---------------- Loaders ----------------
    train_loader = torch.utils.data.DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=2,
        pin_memory=True
    )

    val_loader = torch.utils.data.DataLoader(
        val_ds,
        batch_size=batch_size,
        num_workers=2,
        pin_memory=True
    )

    # ---------------- Model ----------------
    model = KWSCNNTeacherI(
        dropout_fc=dropout_fc,
        freq_mask=freq_mask,
        time_mask=time_mask
    ).to(device)

    # ---------------- Loss ----------------
    criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)

    # ---------------- Optimizer ----------------
    optimizer = optim.Adam(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='max',
        factor=0.5,
        patience=2
    )

    # ---------------- Training ----------------
    best_val_acc = 0
    epochs = 10

    for epoch in range(epochs):

        model.train()

        for x, y in train_loader:

            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            optimizer.zero_grad()

            outputs = model(x)

            loss = criterion(outputs, y)

            loss.backward()

            optimizer.step()

        # ---------------- Validation ----------------
        model.eval()

        correct = 0
        total = 0

        with torch.no_grad():

            for x, y in val_loader:

                x = x.to(device, non_blocking=True)
                y = y.to(device, non_blocking=True)

                outputs = model(x)

                preds = outputs.argmax(1)

                correct += (preds == y).sum().item()
                total += y.size(0)

        val_acc = correct / total

        scheduler.step(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc

        trial.report(val_acc, epoch)

        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return best_val_acc

# =====================================================
# Run Study
# =====================================================

study = optuna.create_study(direction="maximize")

study.optimize(objective, n_trials=30)

# =====================================================
# Best Result
# =====================================================

print("\n✅ Best Trial")
print("Best Accuracy:", study.best_trial.value)

print("\nBest Hyperparameters:")
for key, value in study.best_trial.params.items():
    print(f"{key}: {value}")

[I 2026-04-01 09:19:02,884] A new study created in memory with name: no-name-b53b98fd-7392-4ec1-aca7-81c864285bbd
[I 2026-04-01 09:22:03,608] Trial 0 finished with value: 0.9460952380952381 and parameters: {'lr': 0.0009074311559156162, 'weight_decay': 5.93270801834819e-05, 'label_smoothing': 0.08664824793480871, 'batch_size': 32, 'dropout_fc': 0.3410012137449073, 'freq_mask': 4, 'time_mask': 11}. Best is trial 0 with value: 0.9460952380952381.
[I 2026-04-01 09:24:55,148] Trial 1 finished with value: 0.94 and parameters: {'lr': 0.0009015998060607317, 'weight_decay': 1.0679344006073492e-05, 'label_smoothing': 0.029735025662863988, 'batch_size': 64, 'dropout_fc': 0.34614369200042316, 'freq_mask': 4, 'time_mask': 6}. Best is trial 0 with value: 0.9460952380952381.
[I 2026-04-01 09:27:58,381] Trial 2 finished with value: 0.9506666666666667 and parameters: {'lr': 0.0009186634518293607, 'weight_decay': 9.766742480532984e-05, 'label_smoothing': 0.07635751727336373, 'batch_size': 32, 'dropout_f


✅ Best Trial
Best Accuracy: 0.9611428571428572

Best Hyperparameters:
lr: 0.0008917611716538133
weight_decay: 4.965533835654375e-05
label_smoothing: 0.08473148998314384
batch_size: 16
dropout_fc: 0.26589691465786025
freq_mask: 4
time_mask: 10
